In [1]:
# 🔍 Debug pipeline Silver - Bản không lỗi import

"""Notebook này tự định nghĩa các hàm xử lý (không dùng file gốc) để kiểm tra từng bước pipeline.  
Mục tiêu: phát hiện bước nào gây treo / chậm."""

'Notebook này tự định nghĩa các hàm xử lý (không dùng file gốc) để kiểm tra từng bước pipeline.  \nMục tiêu: phát hiện bước nào gây treo / chậm.'

In [2]:
# 1. Khởi tạo Spark và import thư viện
import os, re, sys, time, unicodedata
from functools import reduce
from datetime import datetime
import unidecode, pandas as pd, numpy as np
from pyspark.sql import SparkSession, DataFrame, Window
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.utils import AnalysisException
import logging
logger = logging.getLogger(__name__)

# Constants
S3_ENDPOINT   = "http://minio:9000"
S3_ACCESS_KEY = "admin"
S3_SECRET_KEY = "password"
BRONZE_BASE = "s3a://warehouse/bronze"
PLATFORMS   = ["careerviet", "topcv", "vietnamworks"]
SILVER_TABLE = "nessie.silver.jobs"
_LOCAL_MODEL_PATH = "/opt/models/hf_cache/careerlake-ner-skill"
NER_MODEL_NAME = _LOCAL_MODEL_PATH if os.path.isdir(_LOCAL_MODEL_PATH) else "zikay3624/careerlake-ner-skill"

SPARK_CONF = {
    "spark.sql.catalog.nessie.ref": "test",
    "spark.sql.extensions": "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    "spark.sql.catalog.nessie": "org.apache.iceberg.spark.SparkCatalog",
    "spark.sql.catalog.nessie.uri": "http://nessie:19120/api/v1",
    "spark.sql.catalog.nessie.authentication.type": "NONE",
    "spark.sql.catalog.nessie.catalog-impl": "org.apache.iceberg.nessie.NessieCatalog",
    "spark.sql.catalog.nessie.s3.endpoint": "http://minio:9000",
    "spark.sql.catalog.nessie.warehouse": "s3://warehouse",
    "spark.sql.catalog.nessie.io-impl": "org.apache.iceberg.aws.s3.S3FileIO",
    "spark.sql.catalog.nessie.cache-enabled": "false",
    "spark.sql.catalog.nessie.s3.access-key-id": "admin",
    "spark.sql.catalog.nessie.s3.secret-access-key": "password",
    "spark.sql.catalog.nessie.s3.path-style-access": "true",
    "spark.driver.extraJavaOptions": "-Daws.region=us-east-1",
    "spark.executor.extraJavaOptions": "-Daws.region=us-east-1",
    "spark.hadoop.fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
    "spark.hadoop.fs.s3a.access.key": "admin",
    "spark.hadoop.fs.s3a.secret.key": "password",
    "spark.hadoop.fs.s3a.endpoint": "http://minio:9000",
    "spark.hadoop.fs.s3a.path.style.access": "true",
    "spark.hadoop.fs.s3a.connection.ssl.enabled": "false",
    "spark.executor.memory": "4g",
    "spark.executor.memoryOverhead": "2g",
    "spark.driver.memory": "4g",
    "spark.sql.shuffle.partitions": "12",
    "spark.default.parallelism": "12",
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
}

ICEBERG_PACKAGES = (
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
    "org.apache.iceberg:iceberg-aws-bundle:1.5.0,"
    "org.apache.hadoop:hadoop-aws:3.3.4"
)

builder = SparkSession.builder.appName("debug_silver")
for k, v in SPARK_CONF.items():
    builder = builder.config(k, v)
spark = builder.config("spark.jars.packages", ICEBERG_PACKAGES).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark OK")

# Helper UDF
def normalize_common(text):
    if not text: return None
    cleaned = re.sub(r"\s+", " ", text, flags=re.UNICODE).strip().lower()
    cleaned = re.sub(r"\(\s+", "(", cleaned)
    cleaned = re.sub(r"\s+\)", ")", cleaned)
    cleaned = re.sub(r"([^\s(])\(", r"\1 (", cleaned)
    return cleaned or None
normalize_common_udf = F.udf(normalize_common, StringType())

Spark OK


In [3]:
# 2. Kiểm tra bảng file_tracker
try:
    df_tracker = spark.table("nessie.meta.file_tracker")
    print("=== file_tracker ===")
    df_tracker.show(10, truncate=False)
    pending = df_tracker.filter(F.col("status") == "pending").count()
    print(f"Pending files: {pending}")
except Exception as e:
    print(f"Lỗi: {e}")

=== file_tracker ===
+--------------------------------------------------------------------+----------+---------+--------------------------+--------------------------+
|file_path                                                           |platform  |status   |created_at                |updated_at                |
+--------------------------------------------------------------------+----------+---------+--------------------------+--------------------------+
|s3a://warehouse/bronze/careerviet/careerviet_20251231_010354.parquet|careerviet|processed|2026-04-14 17:03:33.475497|2026-05-03 13:15:38.066784|
|s3a://warehouse/bronze/careerviet/careerviet_20251231_100522.parquet|careerviet|processed|2026-04-14 17:03:33.475497|2026-05-03 13:15:39.489395|
|s3a://warehouse/bronze/careerviet/careerviet_20260101_010321.parquet|careerviet|processed|2026-04-14 17:03:33.475497|2026-05-03 13:15:40.350405|
|s3a://warehouse/bronze/careerviet/careerviet_20260101_100417.parquet|careerviet|processed|2026-04-14 1

In [4]:
# 3. Lấy 1 file pending
pending_rows = spark.table("nessie.meta.file_tracker") \
    .filter(F.col("status") == "pending") \
    .limit(1) \
    .select("file_path", "platform") \
    .collect()
if not pending_rows:
    print("Không có file pending. Hãy chạy init_tracker.")
    test_file = None
else:
    test_file = pending_rows[0].asDict()
    print(f"Test file: {test_file}")

Test file: {'file_path': 's3a://warehouse/bronze/topcv/topcv_20251103_175302.parquet', 'platform': 'topcv'}


In [10]:
# 4. Các hàm xử lý chính (đã được rút gọn, bỏ NER thật để tránh chậm)
# ĐÃ SỬA LỖI THIẾU job_key

def _prepare_source(df, platform):
    base_cols = ["title","salary","location","experience","expired_date","company",
                 "job_description","level","education","quantity","work_form","skills","category","link"]
    for c in base_cols:
        if c not in df.columns:
            if c == "skills":
                df = df.withColumn(c, F.array().cast(ArrayType(StringType())))
            else:
                df = df.withColumn(c, F.lit(None).cast(StringType()))
    if isinstance(df.schema["location"].dataType, ArrayType):
        df = df.withColumn("location", F.array_join(F.col("location"), ", "))
    if not isinstance(df.schema["skills"].dataType, ArrayType):
        df = df.withColumn("skills", F.array(F.col("skills").cast(StringType())))
    df = df.withColumn("platform", F.lit(platform))
    return df

def load_batch(spark, files):
    dfs = []
    for row in files:
        try:
            df_raw = spark.read.parquet(row["file_path"])
            dfs.append(_prepare_source(df_raw, row["platform"]))
        except Exception as e:
            print(f"Lỗi file {row['file_path']}: {e}")
    return reduce(lambda l,r: l.unionByName(r, allowMissingColumns=True), dfs)

def standardize_jobs(df):
    df = df.withColumn("title_clean", normalize_common_udf(F.col("title"))) \
           .withColumn("location_clean", normalize_common_udf(F.col("location"))) \
           .withColumn("company_clean", normalize_common_udf(F.col("company")))
    # Thêm các cột giả để khớp schema silver (có thể thay bằng parse thật nếu cần)
    df = df.withColumn("min_salary", F.lit(None).cast(LongType())) \
           .withColumn("max_salary", F.lit(None).cast(LongType())) \
           .withColumn("currency", F.lit(None).cast(StringType())) \
           .withColumn("salary_type", F.lit(None).cast(StringType())) \
           .withColumn("min_years", F.lit(None).cast(DoubleType())) \
           .withColumn("max_years", F.lit(None).cast(DoubleType())) \
           .withColumn("experience_type", F.lit(None).cast(StringType())) \
           .withColumn("education_standard", F.lit(None).cast(StringType())) \
           .withColumn("level_standard", F.lit(None).cast(StringType())) \
           .withColumn("work_form_standard", F.lit(None).cast(StringType())) \
           .withColumn("quantity_normalized", F.lit(1.0).cast(DoubleType())) \
           .withColumn("expired_date_norm", F.col("expired_date").cast(DateType()))
    return df.filter(F.col("title_clean").isNotNull() & (F.col("title_clean") != ""))

def enrich_with_ner_and_category(df, spark):
    # Tạm thời bỏ qua NER thật
    df = df.withColumn("tech_skills", F.array().cast(ArrayType(StringType()))) \
           .withColumn("soft_skills", F.array().cast(ArrayType(StringType()))) \
           .withColumn("skills_all", F.array().cast(ArrayType(StringType()))) \
           .withColumn("description", F.col("job_description")) \
           .withColumn("requirement", F.lit("")) \
           .withColumn("category_name_final", F.lit("HOẠT ĐỘNG DỊCH VỤ KHÁC"))
    return df

def normalize_location(df, spark):
    # Giả lập (không thay đổi)
    return df

def _compute_job_key(df):
    job_key_cols = ["link", "title_clean", "expired_date_norm", "processed_date"]
    return df.withColumn(
        "job_key",
        F.sha2(
            F.concat_ws(
                "||",
                *[F.coalesce(F.col(c).cast(StringType()), F.lit("")) for c in job_key_cols]
            ),
            256
        )
    )

def append_to_silver(df, spark):
    spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.silver")
    df = df.withColumn("processed_date", F.current_date())
    df = _compute_job_key(df)
    
    silver_cols = [
        "job_key", "processed_date", "platform", "title_clean", "location_clean",
        "company_clean", "min_salary", "max_salary", "currency", "salary_type",
        "min_years", "max_years", "experience_type", "education_standard",
        "level_standard", "work_form_standard", "quantity_normalized",
        "expired_date_norm", "tech_skills", "soft_skills", "skills_all",
        "category_name_final", "description", "requirement", "link"
    ]
    available_cols = [c for c in silver_cols if c in df.columns]
    df = df.select(*available_cols)
    df.write.format("iceberg").mode("append").saveAsTable(SILVER_TABLE)

def update_status_by_file_path(spark, file_path, status):
    safe_path = file_path.replace("'", "''")
    spark.sql(f"UPDATE nessie.meta.file_tracker SET status = '{status}', updated_at = current_timestamp() WHERE file_path = '{safe_path}'")

print("Đã định nghĩa xong các hàm xử lý (đã sửa lỗi job_key).")

Đã định nghĩa xong các hàm xử lý (đã sửa lỗi job_key).


In [11]:
# 5. Chạy pipeline test với 1 file (không NER)
if test_file:
    files = [test_file]
    print(f"Processing: {test_file['file_path']}")
    try:
        df_union = load_batch(spark, files)
        print(f"load_batch: {df_union.count()} rows")
        
        df_std = standardize_jobs(df_union)
        print(f"standardize_jobs: {df_std.count()} rows")
        
        df_enr = enrich_with_ner_and_category(df_std, spark)
        print(f"enrich: {df_enr.count()} rows")
        
        df_loc = normalize_location(df_enr, spark)
        print(f"normalize_location: {df_loc.count()} rows")
        
        append_to_silver(df_loc, spark)
        print("Append thành công")
        
        update_status_by_file_path(spark, test_file['file_path'], "processed")
        print("Update status OK")
    except Exception as e:
        print(f"Lỗi: {e}")
        import traceback; traceback.print_exc()
else:
    print("Không có file test.")

Processing: s3a://warehouse/bronze/topcv/topcv_20251103_175302.parquet
load_batch: 619 rows
standardize_jobs: 619 rows
enrich: 619 rows
normalize_location: 619 rows
Append thành công
Update status OK


In [12]:
# 6. Kiểm tra kết quả silver
try:
    df_silver = spark.table("nessie.silver.jobs")
    print(f"Số dòng silver: {df_silver.count()}")
    df_silver.select("link", "processed_date").show(5, truncate=False)
except Exception as e:
    print(f"Lỗi: {e}")

Số dòng silver: 10080
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+
|link                                                                                                                                                                                                                                                   |processed_date|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------+
|https://www.topcv.vn/viec-lam/ky-su-ban-hang-sale-engineer-ban-hang-ky-thuat-3-nam-kinh-nghiem-thu-nhap-tu-20-trieu-binh-duong-hai-duong/1923251.html?ta_source=JobSearchList_LinkDeta

In [8]:
print("Hoàn thành debug.")

Hoàn thành debug.
